In [2]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

In [3]:
pwd

'/Users/minhtamninale/Documents/Angela/Spots_troubleshooting/3rd_647_as_PPE/Post_processing'

In [12]:
df = pd.DataFrame()

folder_path = '../Early_embryo'

dfs = []

for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, sep ='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
        dfs.append(df)
        
df = pd.concat(dfs, ignore_index=True)

file_paths =glob.glob('../Early_embryo/*.txt')

for file_path in file_paths:
    file_data = pd.read_csv(file_path, sep='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
    
      
    file_name = file_path.split('/')[-1]
    number = file_name.split('_')[0]
    number = int(number)
    
    file_data['Embryo_ID'] = number
    
    df = pd.concat([df,file_data],ignore_index=True)
    
    main_frame = df.dropna()

In [13]:
main_frame

,x,y,z,conformation,Embryo_ID
771,53.347305,2.739992,2.04,010,40.0
772,84.576606,78.238312,1.92,111,40.0
773,63.547996,83.718295,1.56,111,40.0
774,85.203833,42.519387,1.68,110,40.0
775,26.574617,13.237790,1.68,000,40.0
...,...,...,...,...,...
1537,91.773210,97.253193,2.04,111,73.0
1538,33.969293,16.803081,3.96,111,73.0
1539,29.974847,30.073883,3.60,010,73.0
1540,94.678261,33.507125,3.84,000,73.0


In [14]:
def conformation_meaning(conformation): 
    meanings = { 
        '000': 'Not_touching',
        '100': '3_prime_touch_PPE',
        '010':'5_prime_touch_PPE',
        '001':'3_prime_touch_5_prime',
        '110':'PPE_prime_middle',
        '011':'5_middle',
        '101':'3_prime_middle',
        '111':'All_touching'
    }
    if conformation in meanings: 
        return meanings[conformation]
    else:
        return 'Not supported'
    
main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)

/var/folders/6g/g2plc5bd3jv_sqpbk23jqwm40000gn/T/ipykernel_12459/1598569033.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)


In [15]:
def count_conformation_per_embryo(df):
    counts = {}
    embryo_ids = main_frame['Embryo_ID'].unique()
    for embryo_id in embryo_ids:
        embryo_df = main_frame.loc[main_frame['Embryo_ID'] == embryo_id]
        count_000 = len(embryo_df.loc[embryo_df["conformation"] == '000'])
        count_100 = len(embryo_df.loc[embryo_df["conformation"] == '100'])
        count_010 = len(embryo_df.loc[embryo_df["conformation"] == '010'])
        count_001 = len(embryo_df.loc[embryo_df["conformation"] == '001'])
        count_110 = len(embryo_df.loc[embryo_df["conformation"] == '110'])
        count_011 = len(embryo_df.loc[embryo_df["conformation"] == '011'])
        count_101 = len(embryo_df.loc[embryo_df["conformation"] == '101'])
        count_111 = len(embryo_df.loc[embryo_df["conformation"] == '111'])
        
        counts[embryo_id] = {
            'Counts_of_not_touching':count_000, 
            'Counts_3_prime_touch_PPE': count_100,
            'Counts_5_prime_touch_PPE': count_010,
            'Counts_3_prime_touch_5_prime':count_001,
            'Counts_PPE_prime_middle:':count_110,
            'Counts_5_middle':count_011,
            'Counts_3_prime_middle':count_101,
            'Counts_all_touching':count_111
        }
        
    counts_df = pd.DataFrame(counts).T.reset_index()
    counts_df.columns = ["Embryo_ID", "Counts_of_not_touching","Counts_3_prime_touch_5_prime", 
                        "Counts_5_prime_touch_PPE","Counts_3_prime_touch_PPE","Counts_5_prime_middle",
                        "Counts_PPE_middle","Counts_3_prime_middle","Counts_all_touching"]
    
    return counts_df

In [16]:
counts_conformation = count_conformation_per_embryo(main_frame).sort_values("Embryo_ID")
counts_conformation

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
15,11.0,1,3,2,2,0,0,2,19
3,12.0,2,3,7,11,0,0,1,34
8,14.0,10,6,17,14,0,1,2,22
18,21.0,0,3,1,2,2,0,3,28
12,22.0,5,3,12,7,1,0,0,30
7,25.0,0,2,5,2,0,0,1,19
4,26.0,14,13,21,19,0,3,1,21
10,27.0,1,5,8,8,1,1,0,19
2,33.0,7,10,10,6,0,1,0,25
5,35.0,0,0,3,1,0,1,0,12


In [22]:
df_2 = pd.read_csv('../../2nd_lamin_test/Conformation_results/Early_conformation_conformation_count.csv')
df_2_new = df_2.drop(columns=['Unnamed: 0'])
df_2_new

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,5.0,12,1,11,0,0,0,1,0
1,14.0,66,6,11,1,0,0,0,2
2,4.0,2,0,1,1,0,0,0,1
3,7.0,24,0,18,0,0,0,0,4
4,1.0,2,0,2,0,0,0,0,0
5,15.0,40,0,2,0,0,0,0,1


In [23]:
concatenated_df = pd.concat([df_2_new, counts_conformation])
concatenated_df

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,5.0,12,1,11,0,0,0,1,0
1,14.0,66,6,11,1,0,0,0,2
2,4.0,2,0,1,1,0,0,0,1
3,7.0,24,0,18,0,0,0,0,4
4,1.0,2,0,2,0,0,0,0,0
5,15.0,40,0,2,0,0,0,0,1
15,11.0,1,3,2,2,0,0,2,19
3,12.0,2,3,7,11,0,0,1,34
8,14.0,10,6,17,14,0,1,2,22
18,21.0,0,3,1,2,2,0,3,28


In [24]:
concatenated_df.to_csv('Early_embryo/Early_conformation_conformation_count.csv')

OSError: Cannot save file into a non-existent directory: 'Early_embryo'

In [25]:
def calculate_category_percentages(dataframe):
    #Exclude Embryo ID in calculation
    columns_to_calculate = dataframe.columns[1:]
    
    # Calculate the sum of each row
    row_sums = dataframe[columns_to_calculate].sum(axis=1)

    # Calculate the percentage of each column for each row by dividing the value of each column by the row sum and multiplying by 100
    category_percentages = dataframe[columns_to_calculate].div(row_sums, axis=0) *100
    
    category_percentages.insert(0, 'Embryo_ID', dataframe['Embryo_ID'])
    return category_percentages

In [26]:
percentage_df = calculate_category_percentages(concatenated_df)
percentage_df

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,5.0,48.000000,4.000000,44.000000,0.000000,0.000000,0.000000,4.000000,0.000000
1,14.0,76.744186,6.976744,12.790698,1.162791,0.000000,0.000000,0.000000,2.325581
2,4.0,40.000000,0.000000,20.000000,20.000000,0.000000,0.000000,0.000000,20.000000
3,7.0,52.173913,0.000000,39.130435,0.000000,0.000000,0.000000,0.000000,8.695652
4,1.0,50.000000,0.000000,50.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,15.0,93.023256,0.000000,4.651163,0.000000,0.000000,0.000000,0.000000,2.325581
15,11.0,3.448276,10.344828,6.896552,6.896552,0.000000,0.000000,6.896552,65.517241
3,12.0,3.448276,5.172414,12.068966,18.965517,0.000000,0.000000,1.724138,58.620690
8,14.0,13.888889,8.333333,23.611111,19.444444,0.000000,1.388889,2.777778,30.555556
18,21.0,0.000000,7.692308,2.564103,5.128205,5.128205,0.000000,7.692308,71.794872


In [28]:
percentage_df.to_csv('../Early_embryo/Early_percentages_per_embryo.csv')

In [29]:
transposed_percentage= percentage_df.T
transposed_percentage

,0,1,2,3,4,5,15,3,8,18,...,5,0,1,9,17,6,13,11,14,16
Embryo_ID,5.0,14.000000,4.0,7.000000,1.0,15.000000,11.000000,12.000000,14.000000,21.000000,...,35.000000,40.000000,59.000000,61.0,62.000000,63.000000,65.000000,72.000000,73.000000,74.000000
Counts_of_not_touching,48.0,76.744186,40.0,52.173913,50.0,93.023256,3.448276,3.448276,13.888889,0.000000,...,0.000000,21.621622,4.347826,0.0,13.888889,5.555556,0.000000,19.117647,16.666667,0.000000
Counts_3_prime_touch_5_prime,4.0,6.976744,0.0,0.000000,0.0,0.000000,10.344828,5.172414,8.333333,7.692308,...,0.000000,10.810811,4.347826,0.0,8.333333,16.666667,7.692308,10.294118,20.833333,9.523810
Counts_5_prime_touch_PPE,44.0,12.790698,20.0,39.130435,50.0,4.651163,6.896552,12.068966,23.611111,2.564103,...,17.647059,24.324324,10.869565,10.0,33.333333,22.222222,15.384615,16.176471,20.833333,23.809524
Counts_3_prime_touch_PPE,0.0,1.162791,20.0,0.000000,0.0,0.000000,6.896552,18.965517,19.444444,5.128205,...,5.882353,8.108108,10.869565,0.0,5.555556,11.111111,0.000000,17.647059,4.166667,9.523810
Counts_5_prime_middle,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,5.128205,...,0.000000,2.702703,2.173913,0.0,0.000000,0.000000,0.000000,2.941176,4.166667,4.761905
Counts_PPE_middle,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,1.388889,0.000000,...,5.882353,2.702703,2.173913,0.0,0.000000,0.000000,7.692308,1.470588,0.000000,4.761905
Counts_3_prime_middle,4.0,0.000000,0.0,0.000000,0.0,0.000000,6.896552,1.724138,2.777778,7.692308,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,2.941176,4.166667,0.000000
Counts_all_touching,0.0,2.325581,20.0,8.695652,0.0,2.325581,65.517241,58.620690,30.555556,71.794872,...,70.588235,29.729730,65.217391,90.0,38.888889,44.444444,69.230769,29.411765,29.166667,47.619048


In [30]:
transposed_percentage.to_csv('../Early_embryo/Early_percentages.csv')

In [31]:
transposed_df=concatenated_df.T
transposed_df

,0,1,2,3,4,5,15,3,8,18,...,5,0,1,9,17,6,13,11,14,16
Embryo_ID,5.0,14.0,4.0,7.0,1.0,15.0,11.0,12.0,14.0,21.0,...,35.0,40.0,59.0,61.0,62.0,63.0,65.0,72.0,73.0,74.0
Counts_of_not_touching,12.0,66.0,2.0,24.0,2.0,40.0,1.0,2.0,10.0,0.0,...,0.0,8.0,2.0,0.0,5.0,1.0,0.0,13.0,4.0,0.0
Counts_3_prime_touch_5_prime,1.0,6.0,0.0,0.0,0.0,0.0,3.0,3.0,6.0,3.0,...,0.0,4.0,2.0,0.0,3.0,3.0,1.0,7.0,5.0,2.0
Counts_5_prime_touch_PPE,11.0,11.0,1.0,18.0,2.0,2.0,2.0,7.0,17.0,1.0,...,3.0,9.0,5.0,1.0,12.0,4.0,2.0,11.0,5.0,5.0
Counts_3_prime_touch_PPE,0.0,1.0,1.0,0.0,0.0,0.0,2.0,11.0,14.0,2.0,...,1.0,3.0,5.0,0.0,2.0,2.0,0.0,12.0,1.0,2.0
Counts_5_prime_middle,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,2.0,1.0,1.0
Counts_PPE_middle,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0
Counts_3_prime_middle,1.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0
Counts_all_touching,0.0,2.0,1.0,4.0,0.0,1.0,19.0,34.0,22.0,28.0,...,12.0,11.0,30.0,9.0,14.0,8.0,9.0,20.0,7.0,10.0


In [32]:
def calculate_row_sum(dataframe):
    row_sums = []
    for _, row in dataframe.iterrows():
        row_sum = row.sum()
        row_sums.append(row_sum)
    return row_sums

In [33]:
sum_early = calculate_row_sum(transposed_df)
len(sum_early)

9

In [34]:
def calculate_category_sum(dataframe): 
    categories = ['Embryo_ID','Counts_of_not_touching', 'Counts_3_prime_touch_5_prime', 'Counts_5_prime_touch_PPE',
                  'Counts_3_prime_touch_PPE', 'Counts_5_prime_middle', 'Counts_PPE_middle',
                  'Counts_3_prime_middle', 'Counts_all_touching']    
    row_sums = []
    for _, row in dataframe.iterrows():
        row_sum = row.sum()
        row_sums.append(row_sum)
    row_sums_df = pd.DataFrame({'Category': categories, 'Category_Sum':row_sums})
    return row_sums_df

In [35]:
early_sum = calculate_category_sum(transposed_df).iloc[1:]
early_sum

,Category,Category_Sum
1,Counts_of_not_touching,219.0
2,Counts_3_prime_touch_5_prime,82.0
3,Counts_5_prime_touch_PPE,185.0
4,Counts_3_prime_touch_PPE,101.0
5,Counts_5_prime_middle,10.0
6,Counts_PPE_middle,12.0
7,Counts_3_prime_middle,14.0
8,Counts_all_touching,355.0


In [36]:
transposed_df.to_csv('../Early_embryo/Early_embryo_grouped_conformation.csv')

In [37]:
early_sum.to_csv('../Early_embryo/Early_sum.csv')

In [38]:
def calculate_column_medians(counts_conformation):
    columns_medians = concatenated_df.median()
    columns_median_df = pd.DataFrame(columns_medians, columns=['Median'])
    columns_median_df.reset_index(inplace=True)
    columns_median_df.rename(columns={'index': 'Categories'}, inplace=True)
    return columns_median_df

df_late_median = calculate_column_medians(counts_conformation)
late_embryo_median = df_late_median.drop([0]).reset_index(drop = True)

In [39]:
def calculate_column_medians(counts_conformation):
    columns_medians = concatenated_df.median()
    columns_median_df = pd.DataFrame(columns_medians, columns=['Median'])
    columns_median_df.reset_index(inplace=True)
    columns_median_df.rename(columns={'index': 'Categories'}, inplace=True)
    return columns_median_df

df_early_median = calculate_column_medians(counts_conformation)
early_embryo_median = df_early_median.drop([0]).reset_index(drop = True)

In [40]:
early_embryo_median

,Categories,Median
0,Counts_of_not_touching,2.0
1,Counts_3_prime_touch_5_prime,3.0
2,Counts_5_prime_touch_PPE,5.0
3,Counts_3_prime_touch_PPE,2.0
4,Counts_5_prime_middle,0.0
5,Counts_PPE_middle,0.0
6,Counts_3_prime_middle,0.0
7,Counts_all_touching,12.0


In [42]:
early_embryo_median.to_csv("../Early_embryo/Early_embryo_median_conformation_count.csv")